In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import gc
import json
import optuna
from tqdm.auto import tqdm
from copy import deepcopy

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

from scipy.stats import randint, uniform, loguniform

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from helpers.modeling_helpers import NNModel, get_xgb_pipeline



In [ ]:
train_dataset = pd.read_csv("intermediate_data/train_dataset.csv")
test_dataset = pd.read_csv("intermediate_data/test_dataset.csv")

In [ ]:
train_dataset.drop('Unnamed: 0', axis=1, inplace=True)
test_dataset.drop('Unnamed: 0', axis=1, inplace=True)
train_dataset

In [ ]:
X = train_dataset.drop(["SK_ID_CURR", "TARGET"], axis=1)
y = train_dataset["TARGET"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.to_list()
cat_cols = X_train.columns.difference(num_cols).to_list()
cat_cols
for c in cat_cols:
    X_train[c] = X_train[c].astype("category")
    X_test[c] = X_test[c].astype("category")

# Lightgbm

# Lightgbm Trial

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective_lgbm(trial):
    gc.collect()

    params = {
        "lgbm_model__n_estimators": trial.suggest_int("n_estimators", 300, 3000),
        "lgbm_model__learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "lgbm_model__max_depth": trial.suggest_int("max_depth", 3, 12),
        "lgbm_model__num_leaves": trial.suggest_int("num_leaves", 8, 64),
        "lgbm_model__min_child_samples": trial.suggest_int("min_child_samples", 20, 400),
        "lgbm_model__subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "lgbm_model__subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "lgbm_model__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "lgbm_model__reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "lgbm_model__reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),

        "lgbm_model__random_state": 42,
        "lgbm_model__n_jobs": -1,
        "lgbm_model__verbose": -1,
    }


    lgbm_pipe = Pipeline([
        ("lgbm_model", LGBMClassifier())
    ])

    lgbm_pipe.set_params(**params) 

    scores = cross_val_score(
        lgbm_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1
    )

    return scores.mean()

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=50)

print("Best AUC:", study_lgbm.best_value)
print("Best params:", study_lgbm.best_params)

# XGBoost Trial

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective_xgb(trial):
    gc.collect()
    params = {
        "xgb_model__n_estimators": trial.suggest_int("n_estimators", low=300, high=3000),
        "xgb_model__max_depth": trial.suggest_int("max_depth", low=2, high=20), #####
        "xgb_model__min_child_weight": trial.suggest_float("min_child_weight", low=2, high=12), #####
        "xgb_model__subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "xgb_model__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "xgb_model__learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "xgb_model__gamma": trial.suggest_float("gamma", 0, 5),
        "xgb_model__lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
        "xgb_model__alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),
        "xgb_model__random_state": 42,
        "xgb_model__n_jobs": -1,
    }
    
    xgb_pipe = get_xgb_pipeline(cat_cols)

    xgb_pipe.set_params(**params)

    scores = cross_val_score(
        xgb_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1
    )

    return scores.mean()

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)

print("Best AUC:", study_xgb.best_value)
print("Best params:", study_xgb.best_params)

In [ ]:
print("xgboost best val: ", study_xgb.best_value)
print("lightgbm best val: ", study_lgbm.best_value)

In [ ]:
xgb_best_params = study_xgb.best_params
print(xgb_best_params)

## retrain Xgboost on all of train set

In [ ]:
final_ct = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder='passthrough'
)
xgb_final_pipe = Pipeline([
    ("preprocess", final_ct),
    ("xgb_model", XGBClassifier(**xgb_best_params))
])
xgb_final_pipe.fit(X_train, y_train)

## Save xgboost model and parameters

In [ ]:
import joblib

joblib.dump(xgb_final_pipe, "models/xgb_final_pipeline_w_50_optuna.joblib")

In [ ]:
with open("models/xgb_best_params.json", "w") as f:
    json.dump(study_xgb.best_params, f)
with open("models/lgbm_best_params.json", "w") as f:
    json.dump(study_lgbm.best_params, f)

# Neural Net

In [ ]:
torch_device = "cuda" if torch.cuda.is_available() else 'cpu'
torch_device

### Find all NaN columns and drop

In [ ]:
nan_all_cols = X_train.columns[X_train.isna().all()]
X_train_nn = X_train.drop(columns=nan_all_cols)
X_test_nn = X_test.drop(columns=nan_all_cols)
test_dataset_nn = test_dataset.drop(columns=nan_all_cols)

### Pipeline to process data for NN

In [ ]:
num_cols_nn = X_train_nn.select_dtypes(include=[np.number]).columns.to_list()
cat_cols_nn = X_train_nn.columns.difference(num_cols).to_list()

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", QuantileTransformer(output_distribution="normal"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols_nn),
    ("cat", categorical_pipe, cat_cols_nn)
])

In [ ]:
X_train_preprocessed = preprocess.fit_transform(X_train_nn)
X_test_preprocessed = preprocess.transform(X_test_nn)

### convert DFs to Tensors

In [ ]:
X_train_tensor = torch.tensor(X_train_preprocessed, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_preprocessed,  dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
y_test_tensor  = torch.tensor(y_test.to_numpy(),  dtype=torch.float32).reshape(-1, 1)

In [ ]:
X_train_tensor.shape

In [ ]:
nn_model = NNModel(X_train_tensor.shape[1], dropout_rate=0.4).to(torch_device)

n_pos = y_train_tensor.sum()
n_neg = y_train_tensor.numel() - n_pos
pos_weight = (n_neg / n_pos).to(torch_device)
loss_fun = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(lr=1e-3, params=nn_model.parameters(), weight_decay=1e-4)



In [ ]:
torch_train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
torch_test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

g = torch.Generator()
g.manual_seed(42)
train_dataloader = DataLoader(torch_train_dataset, batch_size=128, shuffle=True, generator=g)
test_loader = DataLoader(torch_test_dataset, batch_size=128, shuffle=False)

In [ ]:
epochs = 50
losses = []
val_losses = []
best_auc = 0
best_state = None
patience = 8
bad = 0


for epoch in tqdm(range(epochs)):
    batch_losses = []
    nn_model.train()
    for X_tr, y_tr in tqdm(train_dataloader, leave=False):

        X_tr = X_tr.to(torch_device)
        y_tr = y_tr.to(torch_device).view(-1, 1)
        y_hat = nn_model(X_tr)
        loss = loss_fun(y_hat, y_tr)

        batch_losses.append(loss.to('cpu').detach())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    val_losses_epoch = []
    nn_model.eval()
    with torch.no_grad():
        ys_list = []
        for X_b, y_b in test_loader:
            X_b = X_b.to(torch_device)
            y_b = y_b.to(torch_device).float().view(-1, 1)

            y_hat_test = nn_model(X_b)
            ys = torch.sigmoid(y_hat_test).cpu().numpy()
            ys_list.append(ys)
            test_loss = loss_fun(y_hat_test, y_b)
            

            val_losses_epoch.append(test_loss.detach().cpu().item())
        ys = np.concatenate(ys_list)
        auc = roc_auc_score(y_test_tensor.numpy(), ys)
        if auc > best_auc:
            best_auc = auc
            best_state = deepcopy(nn_model.state_dict())
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    



    val_losses.append(np.mean(val_losses_epoch))
    
    batch_loss = np.mean(batch_losses)
    losses.append(batch_loss)
nn_model.load_state_dict(best_state)


In [ ]:
plt.plot(losses)

In [ ]:
plt.plot(val_losses)

In [ ]:
nn_model.eval()
preds = []
with torch.no_grad():
    for x_t, y_t in test_loader:
        x_t = x_t.to(torch_device)
        y_t = y_t.to(torch_device)
        y_pr_logits = nn_model(x_t)
        y_pr_sigmoid = torch.sigmoid(y_pr_logits)
        preds.append(y_pr_sigmoid.to('cpu').detach())

y_pred = np.concat(preds)
y_pred


In [ ]:
roc_auc_score(y_test_tensor, y_pred)

In [ ]:
test_dataset_preprocessed = preprocess.transform(test_dataset_nn)
test_dataset_tensor = torch.tensor(test_dataset_preprocessed, dtype=torch.float32).to(torch_device)

In [ ]:
test_dataset_dataset = TensorDataset(test_dataset_tensor)
test_dataset_loader = DataLoader(test_dataset_dataset, batch_size=64, shuffle=False)

In [ ]:
test_preds = []
for x, in test_dataset_loader:
    y_pr = nn_model(x)
    y_pr_sigmoid = torch.sigmoid(y_pr)
    test_preds.append(y_pr_sigmoid.to('cpu').detach())

test_preds = np.concatenate(test_preds)
test_preds

## Predict validation set

In [ ]:
y_proba = xgb_final_pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
auc

In [ ]:
result_df = pd.DataFrame(xgb_final_pipe.predict_proba(test_dataset)[:, 1], columns=['TARGET'])
result_df['SK_ID_CURR'] = test_dataset["SK_ID_CURR"]
result_df.to_csv("./result_xgb_final.csv", index=False)
result_df